# Buddy Brain — YouTube Content Model (Qwen3-4B QLoRA)

**Setup:** Settings → Accelerator → **GPU T4 x2 or P100**, Internet **ON**. Run All.

**Output:** Download `buddy_brain_youtube_outputs.zip` from `/kaggle/working/`  
Put its contents in `brain/incoming/youtube/` on the ASUS repo.

**Time:** ~2-3 hours on T4 x2 for 2 epochs on 10,000 records.

In [ ]:
# 1. Check GPU
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 2. Install Unsloth + dependencies
!pip -q install 'unsloth[kaggle-new]' 2>&1 | tail -5
!pip -q install 'trl>=0.8' 'peft>=0.10' datasets evaluate 2>&1 | tail -3
print('Install done')

In [ ]:
# 3. Clone repo and upload dataset
# OPTION A: Clone from GitHub (repo must be public or you add token)
!git clone --depth 1 https://github.com/kopanathi642/YOUTUBE-BUDDY.git /kaggle/working/repo

# OPTION B: If you uploaded the dataset JSONL files as a Kaggle dataset,
# copy them here instead:
# import shutil
# shutil.copytree('/kaggle/input/buddy-brain-yt-v1', '/kaggle/working/repo/brain/youtube_dataset/yt_v1')

import os
DATASET_DIR = '/kaggle/working/repo/brain/youtube_dataset/yt_v1'
print('Dataset dir exists:', os.path.isdir(DATASET_DIR))
if os.path.isdir(DATASET_DIR):
    for f in os.listdir(DATASET_DIR):
        path = os.path.join(DATASET_DIR, f)
        lines = sum(1 for _ in open(path)) if f.endswith('.jsonl') else '-'
        print(f'  {f}: {lines} records')

In [ ]:
# 4. Load model with Unsloth (4-bit QLoRA — fits in 16GB VRAM)
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
BASE_MODEL  = 'Qwen/Qwen3-4B-Instruct'   # change to Qwen/Qwen3-4B if instruct not available

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,     # auto
    load_in_4bit   = True,
)
print('Model loaded:', BASE_MODEL)

In [ ]:
# 5. Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
model.print_trainable_parameters()

In [ ]:
# 6. Load and format dataset
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_raw = load_jsonl(f'{DATASET_DIR}/train.jsonl')
val_raw   = load_jsonl(f'{DATASET_DIR}/val.jsonl')
print(f'Train: {len(train_raw):,}  Val: {len(val_raw):,}')

# Format as chat template
def format_record(rec):
    text = tokenizer.apply_chat_template(
        rec['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_ds = Dataset.from_list([format_record(r) for r in train_raw])
val_ds   = Dataset.from_list([format_record(r) for r in val_raw])
print('Sample:', train_ds[0]['text'][:300])

In [ ]:
# 7. Train!
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUT_DIR = '/kaggle/working/buddy-brain-yt-v1'

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    dataset_text_field = 'text',
    max_seq_length  = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps        = 50,
        num_train_epochs    = 2,
        learning_rate       = 2e-4,
        fp16                = not is_bfloat16_supported(),
        bf16                = is_bfloat16_supported(),
        logging_steps       = 20,
        evaluation_strategy = 'steps',
        eval_steps          = 100,
        save_strategy       = 'steps',
        save_steps          = 200,
        output_dir          = OUT_DIR,
        optim               = 'adamw_8bit',
        weight_decay        = 0.01,
        lr_scheduler_type   = 'linear',
        seed                = 42,
    ),
)

trainer_stats = trainer.train()
print('Training done:', trainer_stats)

In [ ]:
# 8. Quick test before saving
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_messages = [{
    'role': 'system',
    'content': 'You are Buddy Brain, an expert YouTube content strategist for Indian creators. Always return valid JSON only.'
}, {
    'role': 'user',
    'content': 'Create YouTube content for:\nNiche: cooking\nKeywords: dal recipe, easy, 10 minutes\nChannel type: food & cooking\nTarget views: growing (50K-500K views)'
}]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt'
).to('cuda')

outputs = model.generate(
    input_ids   = inputs,
    max_new_tokens = 400,
    temperature = 0.4,
    do_sample   = True,
)
answer = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print('Model answer:')
print(answer)

# Validate JSON
import json
try:
    parsed = json.loads(answer)
    print('\n✅ Valid JSON! Keys:', list(parsed.keys()))
except:
    print('\n⚠️  Not valid JSON — model needs more training data')

In [ ]:
# 9. Save LoRA adapter + convert to GGUF for Ollama
import os

ADAPTER_DIR = '/kaggle/working/buddy-brain-yt-v1-adapter'
GGUF_DIR    = '/kaggle/working/buddy-brain-yt-v1-gguf'

# Save LoRA adapter (small, ~100MB)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Adapter saved to', ADAPTER_DIR)

# Convert to GGUF Q4_K_M (runs on Ollama, ~2.5GB)
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method = 'q4_k_m',
)
print('GGUF saved to', GGUF_DIR)

# List output files
for root, dirs, files in os.walk('/kaggle/working'):
    for f in files:
        fp = os.path.join(root, f)
        size_mb = os.path.getsize(fp) / 1e6
        if size_mb > 1:
            print(f'  {fp}: {size_mb:.1f} MB')

In [ ]:
# 10. Package everything for download
!cd /kaggle/working && zip -r buddy_brain_youtube_outputs.zip \
    buddy-brain-yt-v1-adapter \
    buddy-brain-yt-v1-gguf \
    buddy-brain-yt-v1/trainer_log.jsonl \
    2>&1 | tail -5

import os
zip_size = os.path.getsize('/kaggle/working/buddy_brain_youtube_outputs.zip') / 1e6
print(f'\n✅ buddy_brain_youtube_outputs.zip: {zip_size:.0f} MB')
print('\nNext steps:')
print('1. Download buddy_brain_youtube_outputs.zip from Output tab')
print('2. Extract to brain/incoming/youtube/ on ASUS')
print('3. Run: ollama create buddy-brain -f brain/kaggle_youtube/Modelfile')
print('4. Run: ollama run buddy-brain')